# Parameter estimation using Bayesian inference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In this notebook, we will give a quick overview of how to perform bayesian inference on a simple signal, and predict the parameters associated with that signal.

To do this we would like to predict the probability that a set of parameters $\theta$ is associated with our observed data $d$, $p(\theta | d)$. 

First lets define a few things, the model we will use for the signal can just be a straight line, however this can be replaced with many other models. This model is defined by
$$
y = mx + c.
$$


In [ ]:
def signal_model(x, m, c):
    """simple straight line model"""
    return m*x+c

We can create a fake piece of data by taking our model and adding some random Gaussian noise. This essentially means taking N random draws from a Gaussian distribution with some variance $\sigma$ and adding them to a simulated signal.

In [ ]:
N = 10                            # number of data points
variance = 0.001                   # variance of noise
sigma = np.sqrt(variance)
x = np.linspace(0,1,N)            # create x positions
model = signal_model(x, 1, 0)     # create line with gradient 1 and 0 intercept

data = model + np.random.normal(0, sigma, size=N)

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, data, label="data")
ax.plot(x, model, label="signal")
ax.legend()
ax.set_xlabel("xposition")
ax.set_ylabel("yposition")

Given this piece of data, we would like to predict the parameters $\theta = [m, c]$ (assuming we dont already know them). To do this we can use [Bayes' theorem](https://en.wikipedia.org/wiki/Bayes%27_theorem):
$$
p(\theta | d) = \frac{p(\theta)p(d | \theta)}{p(d)}.
$$
This returns the probability that parameters $\theta$ are consistent with the data $d$, also know as the **posterior** distribution on parameters $\theta$. Within this there are three other distributions: the **prior** $p(\theta)$, the **likelihood** $p(d|\theta)$ and the **evidence** $p(d)$. 
The prior distribution describes any knowledge you have about the probability of a parameter before this experiment.
The likelihood describes the probability of observing data $d$ given the parameters $\theta$.
The evidence is the intergral of the numerator over all available parameters and can often be ignored as a normalisation constant (though it is useful when comparing different models), leaving
$$
p(\theta | d) \propto p(\theta)p(d | \theta).
$$

Next up we can define a likelihood function, this encapuslates out understanding of both the signal and the noise models. For simplicity (and is correct in many cases) we can assume that our signal is in additive Gaussian noise. This allows us to use the Gaussian distribution as out likelihood function. You may have come across something similar to this doing any sort of least squares fitting. We know this is the correct likelihood in this case as we simulated Gaussian noise above.

The Gaussian distribution is defined by
$$
p(x | \mu, \sigma) = \frac{1}{\sqrt{2\pi \sigma^2}} \exp{\left(- \frac{(x - \mu)^2}{2\sigma^2}\right)}
$$
We often write this out as a log likelihood as log space is easier for a computer to handle (less overflows etc).

In [ ]:
def log_likelihood(params, data, x, sigma):
    """Gaussian log likelihood function
    Arguments
    ----------
    params: list
        list of model parameters to predict
    data: np.array
        measured data
    x: np.array
        x positions of data points
    
    Returns
    --------
    np.array: log_likelihood
    """
    m, c = params

    model = signal_model(x, m, c)

    # compute normalising factor for a log Gaussian distribution
    norm_factor = -0.5 * np.log(2*np.pi*sigma*sigma)

    # compute exponent of a log gaussian distribution
    exponent = -(data - model)**2 / (2*sigma*sigma)

    return norm_factor + np.sum(exponent)

If we assume that the prior is flat, i.e. its a fixed value for all parameter values, then Bayes' theorem becomes
$$
p(\theta | d) \propto p(d | \theta)
$$

We now have a simple distribution to compute, and in low dimensions we can directly compute this for all parameter values in some range.

In [ ]:
ngrid = 100
mvalues = np.linspace(0.8,1.2,ngrid)

# loop over every point in the grid and compute log likelihood
log_likelihood_grid = np.zeros(ngrid)
for i in range(ngrid):
    params = (mvalues[i], 0) 
    log_likelihood_grid[i] = log_likelihood(params, data, x, sigma)

In [ ]:
plt.figure()
plt.plot(mvalues, np.exp(log_likelihood_grid))
plt.axvline(1.0,color='r',label='$m=1$')
plt.legend()
plt.show()

Technically, this is the likelihood, but to convert it to a posterior (assuming a flat prior on $m$) you just need to normalise it.

**Write a function that jointly infers the slope and intercept at the same time.** HINT: you may want to reduce the number of grid points you use, as this could be slow... (we'll come to ways to get around this later)

# Parameter estimation mock data challenge

## Part 1: estimating the primary mass of a gravitational wave event

The same principle that is used for finding the slope and intercept of a line can also be used to infer the properties of a gravitational wave signal, including the masses of the primary and secondary component of the merger ($m_1$ and $m_2$), its luminosity distance ($d_L$), sky localisation, spins, and many other parameters.

There are three files on the data folder
 - **PE_singledet_unblind_data.txt** which contains the times and strains for a chunk of gravitational wave detector data which includes a single gravitational-wave signal at a merger time of ~1.13s.
 - **PE_singledet_unblind_noise.txt** which contains the times and strains for a chunk of empty (noise-only) gravitational wave detector data (this can be used to compute the PSD more accurately than by using a chunk of data which contains a signal).
 - **PE_singledet_unblind_parameters.txt** which contains the real true source parameters for the event - this is what you're hoping to find out!

**Write a new log likelihood function for a gravitational wave signal which estimates the primary mass of the signal.** Do you get the right value? (Check the PE_signal_parameters file to see).

This will require
1. Simulation of a signal/template
2. Computation of the PSD
3. Computation of likelihood over parameter space

We can write a similar Gaussian likelihood function as seen before, but using the PSD in the frequency domain:

\begin{equation}
\log L \propto -\frac{2}{T} \sum_k \frac{|\tilde{d}(f_k) - \tilde{h}(f_k) |^2}{S(f_k)}
\end{equation}

Hint: dont forget the factor of $\Delta t$ in the fourier transforms (see the signal processing notebook), $\tilde{d}(f_k) = \tilde{d}_k \Delta t$, where $\tilde{d}_k$ is the output of np.fft.rfft

See the appendix of [Veitch & Vecchio (2010)](https://arxiv.org/pdf/0911.3820) for more details.


## Part 2: using MCMC to tackle higher dimensions

Now when the dimensions of the problem begin to increase and the size of each parameter space increases, it becomes infeasible to compute the likelihood on a grid. Computing on a grid means spending much of the computation time in regions of low likelihood which isn't very efficient. 

This is where we turn to techniques which can search through this space more efficiently. One of those techniques is MCMC or [Markov Chain Monte-Carlo](https://en.wikipedia.org/wiki/Markov_chain_Monte_Carlo).

**Use the popular mcmc library *emcee* to infer the posterior for the analysis above.** You can start by following the 'basic usage' example in the documentation: https://emcee.readthedocs.io/en/stable/ and adapting it to use your `log_likelihood` function.

In [ ]:
import emcee

Once you have successfully inferred 1 parameter, **modify your analysis to jointly estimate primary mass, secondary mass, and luminosity distance simultaneously**.

## Part 3: inferring the parameters of unknown gravitational wave detections

In the data folder you will find files labelled PE_singledet_blind_data.txt. This contains a gravitational wave signal with an unknown set of parameters. **Apply your parameter estimation algorithm to estimate the masses and distances of these events**. What have you discovered? Check your answers with a demonstrator!

## Part 4: Now it's up to you

Once you've got this far, congratulations! You now know the basics of gravitational wave parameter estimation. Use the rest of your time in the labs to investigate an aspect of gravitational wave parameter estimation more deeply. For example, you might like to:

- If you're working with a lab partner who has successfully detected some gravitational wave events, use your algorithm to estimate the parameters of those events 
- Try estimating a different set of parameters, such as the sky location (RA, $\delta$) of the event. Multiple detectors help to triangulate a signal on the sky, so you'll have to incorporate data from a second gravitational wave detector (speak to a demonstrator for access to the additional data). How do your results change?
- Try out your parameter estimation algorithm on a real gravitational wave event!
- Perhaps you have some other ideas of your own...?

Speak to a demonstrator to discuss ideas.
